# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any

from IPython.display import display



In [3]:
# Load config
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))



home directory: /gpfs01/euler/User/ssuhai
Config path: GitRepos/simulation_closed_loop/config


Configuration loaded successfully:
data_subfolders:
  day: 20250717
  experiment: 1
DJ:
  username: ssuhai
  userinfo:
    experimenter: closedlooptest
    animal_loc: 1
    region_loc: 2
    field_loc: 3
    stimulus_loc: 4
    cond1_loc: 5
    data_dir: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_rfs
  table_parameters:
    PreprocessParams:
      fs_resample: 60
    Stimulus:
      noise:
        stim_name: densenoise
        stim_family: noise
        pix_n_x: 20
        pix_n_y: 15
        skip_duplicates: true
        pix_scale_x_um: 40
        pix_scale_y_um: 40
        framerate: 5
    DNoiseTraceParams:
      dnoise_params_id: 1
      fupsample_trace: 20
      fupsample_stim: 4
      ref_time: stim
      fit_kind: gradient
      skip_duplicates: true
      pre_blur_sigma_s: 0.0
      post_blur_sigma_s: 0.0
paths:
  repo_directory: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/
  home_directory: /gpfs01/euler/User/ssuhai
  dj_confi

In [26]:
from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper
from simulations.loop_components.recording_file_copier import copy_rec_files,create_directory_structure
from simulations.loop_components.stimulus import create_rf_avi_from_roi_ids, create_full_avi_from_roi_id_and_seed
from simulations.loop_components.utils import log

### Create processing components (connect them to DB)

In [ ]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )



In [18]:

# # Load config and tables
# dj_table_holder.load_config()
# dj_table_holder.load_tables()
# print(" loaded and configured successfully")
# dj_table_holder.clear_tables("all")

dj_table_holder.setup()




schema_name: ageuler_ssuhai_closed_loop


/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:195: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:203: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:112: UserWarning: Values for ['bardx', 'bardy', 'velumsec', 'tmovedurs'] in `stim_dict` for stimulus `movingbar` are None. This may cause problems downstream.
  warnings.warn(f'Values for {missing_info} in `stim_dict` for stimulus `{stim_name}` are None. '
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:39: UserWarning: Number of triggers in trial_info=8 must match ntrigger_rep=1.
  warnings.warn(msg)


preprocessing params:
 {'fs_resample': 60}
Saving classifier to /gpfs01/euler/User/ssuhai/datajoint/rgc_classifier/rgc_classifier.pkl


In [19]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)


## During the experimet

### Move files from server to the repo 

In [20]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,
                           date=  cfg.data_subfolders.day, 
                           experiment = cfg.data_subfolders.experiment)

copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    date=cfg.data_subfolders.day,  # type: ignore
    experiment=cfg.data_subfolders.experiment,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

SKIPPING File M1_RR_od.smh: does not match any permissible stimulus type.
SKIPPING File M1_RR_od.smp: does not match any permissible stimulus type.
COPIED file from /gpfs01/euler/data/Data/Suhai/move_closed_loop_data_here/M1_RR_GCL1_chirp_1.smp to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_rfs/20250717/1/Raw/M1_RR_GCL1_chirp_1_iter0.smp
COPIED file from /gpfs01/euler/data/Data/Suhai/move_closed_loop_data_here/M1_RR_GCL1_MB_1.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_rfs/20250717/1/Raw/M1_RR_GCL1_MB_1_iter0.smh
COPIED file from /gpfs01/euler/data/Data/Suhai/move_closed_loop_data_here/M1_RR_GCL1_chirp_1.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_rfs/20250717/1/Raw/M1_RR_GCL1_chirp_1_iter0.smh
COPIED file from /gpfs01/euler/data/Data/Suhai/move_closed_loop_data_here/M1_RR_GCL1_DN_1.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_r

In [21]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/test_rfs/20250717/1
		header_name: 20250717_left.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 7, 17, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 29.11it/s]


Found 3 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}
	Adding field: `{'field': 'GCL1', 'region': 'RR', 'cond1': '1', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 6/6 [00:03<00:00,  1.57it/s]


In [22]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
field_key = missing_keys[0]
missing_keys
# import datetime
# test_key = {'experimenter': 'closedlooptest',
#   'date': datetime.date(2025, 7, 17),
#   'exp_num': 1,
#   'raw_id': 1,
#   'field': 'GCL1',
#   'region': 'RR',
#   'cond1': 'iter0'}
# missing_keys = [test_key]

[{'experimenter': 'closedlooptest',
  'date': datetime.date(2025, 7, 17),
  'exp_num': 1,
  'raw_id': 1,
  'field': 'GCL1',
  'region': 'RR',
  'cond1': '1'}]

In [23]:
# somehow I get a circular import error if I import this at the top
from simulations.gui.integrated_autorois import InteractiveRoiCanvas

In [ ]:
online_analysis_gui = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper],
    field_key=field_key,
    canvas_width=30,
    )

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml


In [25]:
display(online_analysis_gui.start_gui())

# The stimulus

In [ ]:
roi_ids = []


In [ ]:
# import numpy as np
# all_roi_ids = dj_table_holder("PeakSTAPosition")().fetch("roi_id")
# np.random.seed(123)
# roi_ids = np.random.choice(all_roi_ids, size=10, replace=False).tolist()
# print(f"Selected ROI IDs: {roi_ids}")


Selected ROI IDs: [58, 31, 68, 102, 98, 95, 9, 6, 1, 67]


In [31]:
create_rf_avi_from_roi_ids(roi_ids= roi_ids,
                           peak_sta_position_table= dj_table_holder("PeakSTAPosition"),
                           abs_save_dir=os.path.join(cfg.paths.stimulus_output_dir))
log(f"Created RF AVI for ROI IDs: {roi_ids}")

Found 2 rois in the PeakSTAPosition table.
Creating RF test stimulus tensor...
Saving RF test stimulus metadata to  /gpfs01/euler/data/Data/Suhai/stimuli_closed_loop/rf_test_stimulus_metadata.txt
Position data saved to /gpfs01/euler/data/Data/Suhai/stimuli_closed_loop/rf_test_stimulus_metadata.txt
Creating RF test stimulus avi file at  /gpfs01/euler/data/Data/Suhai/stimuli_closed_loop/rf_test_stimulus.avi
DONE!


# Clean up

In [32]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")


[2025-08-12 18:21:27,897][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__field__stack_averages`
[2025-08-12 18:21:27,964][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__presentation__scan_info`
[2025-08-12 18:21:28,010][INFO]: Deleting 9 rows from `ageuler_ssuhai_closed_loop`.`__presentation__stack_averages`
[2025-08-12 18:21:28,150][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__peak_s_t_a_position`
[2025-08-12 18:21:28,200][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a__data_set`
[2025-08-12 18:21:28,253][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-08-12 18:21:28,290][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-08-12 18:21:28,323][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__d_noise_trace`
[2025-08-12 18:21:28,465][INFO]: Deleting 95 rows from `ageuler_ssuhai_closed_loop`.`__celltype_assignment`
[2025-08-12 18:21:28,496][INFO]: Deleting 95 rows 